<a href="https://colab.research.google.com/github/tzt425-gif/AI_AH2179/blob/main/applied_ai_in_transport_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from google.colab import drive

# 1. 挂载 Google Drive
drive.mount('/content/drive')

# 2. 定义绝对路径（与你的 Drive 文件夹名称完全匹配）
BASE_DIR = '/content/drive/MyDrive/AH2179_AI_Transport_Project'
DATA_DIR = os.path.join(BASE_DIR, 'data/processed')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
FIGURES_DIR = os.path.join(BASE_DIR, 'figures')

# 3. 检查数据文件
file_path = os.path.join(DATA_DIR, 'master_feature_table_line4.parquet')

if os.path.exists(file_path):
    print("✅ 成功连接至数据集！")
    df = pd.read_parquet(file_path)
    print(f"数据总行数: {df.shape[0]:,}, 特征列数: {df.shape[1]}")
else:
    print("⚠️ 路径正确但未找到文件，请检查 member A 是否已将 master_feature_table_line4.parquet 上传至 data/processed/ 目录下。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 成功连接至数据集！
数据总行数: 195,524, 特征列数: 44


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import r2_score
from sklearn.preprocessing import RobustScaler
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import LSTM, BatchNormalization, Dense, Dropout
from tensorflow.keras.models import Sequential

# ---------------------------------------------------------
# 0. 目录配置
# ---------------------------------------------------------
MODELS_DIR = './models' if 'MODELS_DIR' not in globals() else MODELS_DIR
DATA_DIR = './data' if 'DATA_DIR' not in globals() else DATA_DIR
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ---------------------------------------------------------
# 1. 特征定义 (严格与队友 Temporal 与 Spatial 特征清单对齐)
# ---------------------------------------------------------
FEATURE_COLS_TEMPORAL = [
    'delay_lag_1',
    'delay_lag_2',
    'delay_diff_prev',
    'scheduled_arrival_time_sfm',
    'hour_sin',
    'hour_cos',
    'day_of_week',
    'is_weekend',
    'is_morning_peak',
    'is_evening_peak',
    'stop_sequence',
    'dist_traveled',
]

FEATURE_COLS_SPATIAL = FEATURE_COLS_TEMPORAL + [
    'network_length_m',
    'num_traffic_signals',
    'num_intersections',
    'bus_lane_ratio',
    'has_bus_priority',
]

TARGET_COL = 'observed_arrival_delay'

# 检查特征列完整性
missing_cols = [col for col in FEATURE_COLS_SPATIAL if col not in df.columns]
if missing_cols:
    print(f'❌ 缺失列: {missing_cols}')
else:
    print(
        f'✅ 特征列校验 100% 通过！纯时序特征数 = {len(FEATURE_COLS_TEMPORAL)} 维 |'
        f' 时空增强特征数 = {len(FEATURE_COLS_SPATIAL)} 维'
    )

# ---------------------------------------------------------
# 2. 严格按队友要求的 Temporal Split (Jan 1–18 / Jan 19–23 / Jan 24–31)
# ---------------------------------------------------------
# 假设 date 格式为数值型 YYYYMMDD (如 20240118) 或可直接比较格式
train_df = df[(df['date'] >= 20240101) & (df['date'] <= 20240118)].copy()
val_df   = df[(df['date'] >= 20240119) & (df['date'] <= 20240123)].copy()
test_df  = df[(df['date'] >= 20240124) & (df['date'] <= 20240131)].copy()

print(f"📊 数据划分完成: Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")


# ---------------------------------------------------------
# 3. 严谨 3D 序列构建函数 (仅在 Train fit，防泄露 + 按 ['date', 'trip_id', 'direction_id'] 分组)
# ---------------------------------------------------------
def prepare_3d_tensors(feature_cols, sequence_length=3):
    scaler = RobustScaler()
    train_scaled = train_df.copy()
    val_scaled   = val_df.copy()
    test_scaled  = test_df.copy()

    # 严谨拟合：仅在 Train 上 fit，在 Val 和 Test 上 transform
    train_scaled[feature_cols] = scaler.fit_transform(train_scaled[feature_cols])
    val_scaled[feature_cols]   = scaler.transform(val_scaled[feature_cols])
    test_scaled[feature_cols]  = scaler.transform(test_scaled[feature_cols])

    def build_correct_sequences(df_data):
        X, y = [], []
        # 队友核心要求：序列分组统一使用 ['date', 'trip_id', 'direction_id']
        group_cols = ['date', 'trip_id', 'direction_id']

        for _, group in df_data.groupby(group_cols):
            group = group.sort_values('stop_sequence')
            features = group[feature_cols].values
            targets = group[TARGET_COL].values

            n_samples = len(features)
            if n_samples > 0:
                for i in range(n_samples):
                    if i < sequence_length - 1:
                        pad_len = sequence_length - (i + 1)
                        pad_features = np.tile(features[0], (pad_len, 1))
                        seq_features = np.vstack([pad_features, features[: i + 1]])
                    else:
                        seq_features = features[i - sequence_length + 1 : i + 1]

                    X.append(seq_features)
                    y.append(targets[i])

        return np.array(X), np.array(y)

    X_tr, y_tr = build_correct_sequences(train_scaled)
    X_va, y_va = build_correct_sequences(val_scaled)
    X_te, y_te = build_correct_sequences(test_scaled)

    return X_tr, y_tr, X_va, y_va, X_te, y_te


# ---------------------------------------------------------
# 4. 标准化 LSTM 训练与评估流程 (固定 Seed 确保可复现 & 重置 Weights)
# ---------------------------------------------------------
def train_and_evaluate(X_tr, y_tr, X_va, y_va, X_te, y_te, exp_name, save_filename, seed=42):
    # 固定随机种子，保证完全可复现
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.backend.clear_session()

    model = Sequential([
        LSTM(
            64,
            return_sequences=True,
            input_shape=(X_tr.shape[1], X_tr.shape[2]),
        ),
        BatchNormalization(),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        BatchNormalization(),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1, activation='linear'),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.Huber(delta=10.0),
        metrics=['mae'],
    )

    save_path = os.path.join(MODELS_DIR, save_filename)
    # 严格使用独立的 Validation 集做 EarlyStopping 和保存最佳模型
    callbacks = [
        EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True
        ),
        ModelCheckpoint(save_path, monitor='val_loss', save_best_only=True),
    ]

    print('\n' + '=' * 55)
    print(f'🚀 开始训练实验: {exp_name}')
    print(f'特征输入维度: {X_tr.shape[2]} 维 | 训练集样本数: {X_tr.shape[0]} | 验证集样本数: {X_va.shape[0]}')
    print('=' * 55)

    model.fit(
        X_tr,
        y_tr,
        validation_data=(X_va, y_va),
        epochs=25,
        batch_size=128,
        callbacks=callbacks,
        verbose=1,
    )

    # 在独立的 Test 集上做最终评估
    y_pred = model.predict(X_te).flatten()
    mae = np.mean(np.abs(y_te - y_pred))
    rmse = np.sqrt(np.mean((y_te - y_pred) ** 2))
    r2 = r2_score(y_te, y_pred)

    return mae, rmse, r2


# ---------------------------------------------------------
# 5. 执行两次独立消融实验 (同一 Seed，保证模型独立重新初始化)
# ---------------------------------------------------------
# 实验 3a: LSTM Temporal Baseline (12 维)
X_tr_temp, y_tr_temp, X_va_temp, y_va_temp, X_te_temp, y_te_temp = prepare_3d_tensors(
    FEATURE_COLS_TEMPORAL
)
mae_temp, rmse_temp, r2_temp = train_and_evaluate(
    X_tr_temp, y_tr_temp,
    X_va_temp, y_va_temp,
    X_te_temp, y_te_temp,
    'Model 3a: Deep LSTM Temporal Baseline (12 features)',
    'best_lstm_temporal_12d.keras',
    seed=42
)

# 实验 3b: LSTM Spatial-Enhanced (17 维)
X_tr_spat, y_tr_spat, X_va_spat, y_va_spat, X_te_spat, y_te_spat = prepare_3d_tensors(
    FEATURE_COLS_SPATIAL
)
mae_spat, rmse_spat, r2_spat = train_and_evaluate(
    X_tr_spat, y_tr_spat,
    X_va_spat, y_va_spat,
    X_te_spat, y_te_spat,
    'Model 3b: Deep LSTM Spatial-Enhanced (17 features)',
    'best_lstm_spatial_17d.keras',
    seed=42
)

# ---------------------------------------------------------
# 6. 100% 对齐队友表格结构的最终对比总表 (全量数据，含极端延误)
# ---------------------------------------------------------
master_data = [
    {
        'Model': 'Persistence',
        'Feature Set': 'Previous-stop delay only',
        'MAE (sec)': 38.679,
        'MAE (min)': 0.645,
        'RMSE (sec)': 60.564,
        'RMSE (min)': 1.009,
        'R²': 0.9389,
    },
    {
        'Model': 'Ridge',
        'Feature Set': 'Temporal + Schedule',
        'MAE (sec)': 37.195,
        'MAE (min)': 0.620,
        'RMSE (sec)': 57.224,
        'RMSE (min)': 0.954,
        'R²': 0.9454,
    },
    {
        'Model': 'Ridge',
        'Feature Set': 'Temporal + Spatial',
        'MAE (sec)': 36.485,
        'MAE (min)': 0.608,
        'RMSE (sec)': 54.599,
        'RMSE (min)': 0.910,
        'R²': 0.9503,
    },
    {
        'Model': 'LightGBM',
        'Feature Set': 'Temporal + Schedule',
        'MAE (sec)': 25.669,
        'MAE (min)': 0.428,
        'RMSE (sec)': 99.516,
        'RMSE (min)': 1.659,
        'R²': 0.8349,
    },
    {
        'Model': 'LightGBM',
        'Feature Set': 'Temporal + Spatial',
        'MAE (sec)': 23.560,
        'MAE (min)': 0.393,
        'RMSE (sec)': 98.075,
        'RMSE (min)': 1.635,
        'R²': 0.8397,
    },
    {
        'Model': 'LSTM',
        'Feature Set': 'Temporal + Schedule',
        'MAE (sec)': round(mae_temp, 3),
        'MAE (min)': round(mae_temp / 60, 3),
        'RMSE (sec)': round(rmse_temp, 3),
        'RMSE (min)': round(rmse_temp / 60, 3),
        'R²': round(r2_temp, 4),
    },
    {
        'Model': 'LSTM',
        'Feature Set': 'Temporal + Spatial',
        'MAE (sec)': round(mae_spat, 3),
        'MAE (min)': round(mae_spat / 60, 3),
        'RMSE (sec)': round(rmse_spat, 3),
        'RMSE (min)': round(rmse_spat / 60, 3),
        'R²': round(r2_spat, 4),
    },
]

df_master_all = pd.DataFrame(master_data)

print('\n' + '=' * 65)
print('🎉 100% 格式对齐的 Master Ablation Matrix (消融矩阵总表)')
print('=' * 65)
display(df_master_all)

# 计算 LSTM 空间特征增益
mae_diff = mae_temp - mae_spat
mae_gain = (mae_diff / mae_temp) * 100

print('\n💡 你跑出的最新 LSTM 空间特征增益分析 (Ablation Gain):')
print(
    f'• Temporal MAE: {mae_temp:.3f} s ({mae_temp/60:.3f} min)  |  Spatial MAE:'
    f' {mae_spat:.3f} s ({mae_spat/60:.3f} min)'
)
print(
    f'• Temporal RMSE: {rmse_temp:.3f} s | Spatial RMSE: {rmse_spat:.3f} s |'
    f' R²: {r2_spat:.4f}'
)

if mae_diff > 0:
    print(
        f'✅ Spatial 特征起到了正面作用！MAE 降低了 {mae_diff:.3f} 秒 (优化率:'
        f' {mae_gain:.2f}%)'
    )
else:
    print(
        f'⚠️ Spatial MAE 变化为 {mae_diff:.3f}'
        ' 秒。如出现微幅波动，这是由于测试集中包含极端延误离群值所致。'
    )

# 导出 CSV 供直接使用
output_csv_path = os.path.join(DATA_DIR, 'master_model_ablation_comparison.csv')
df_master_all.to_csv(output_csv_path, index=False)
print(f"\n✅ 对比总表已成功导出至: {output_csv_path}")

✅ 特征列校验 100% 通过！纯时序特征数 = 12 维 | 时空增强特征数 = 17 维
📊 数据划分完成: Train=115164 | Val=27666 | Test=52694


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



🚀 开始训练实验: Model 3a: Deep LSTM Temporal Baseline (12 features)
特征输入维度: 12 维 | 训练集样本数: 115164 | 验证集样本数: 27666
Epoch 1/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - loss: 738.9549 - mae: 78.6659 - val_loss: 436.9688 - val_mae: 48.3794
Epoch 2/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - loss: 415.2664 - mae: 46.2235 - val_loss: 374.1446 - val_mae: 42.0682
Epoch 3/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 388.1055 - mae: 43.4964 - val_loss: 356.5708 - val_mae: 40.2748
Epoch 4/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 369.0460 - mae: 41.5756 - val_loss: 331.2923 - val_mae: 37.7292
Epoch 5/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 354.1723 - mae: 40.0797 - val_loss: 324.9887 - val_mae: 37.0971
Epoch 6/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - loss: 345.0958 - mae: 39.1625 - val_loss: 306.1062 - val_mae: 35.2034
Epoch 7/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 334.9940 - mae: 38.1452 - val_loss: 295.3915 - val_mae: 34.0953
Epoch 8/25
900/9

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


900/900 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - loss: 696.1438 - mae: 74.3551 - val_loss: 349.7433 - val_mae: 39.5177
Epoch 2/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 348.8302 - mae: 39.5258 - val_loss: 288.5429 - val_mae: 33.3877
Epoch 3/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 325.9869 - mae: 37.2224 - val_loss: 275.8586 - val_mae: 32.1239
Epoch 4/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 308.0157 - mae: 35.4053 - val_loss: 262.7188 - val_mae: 30.7687
Epoch 5/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 295.5799 - mae: 34.1423 - val_loss: 252.7246 - val_mae: 29.7172
Epoch 6/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 288.5242 - mae: 33.4315 - val_loss: 249.2398 - val_mae: 29.3804
Epoch 7/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 281.7923 - mae: 32.7494 - val_loss: 264.0964 - val_mae: 30.8673
Epoch 8/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 274.8384 - mae: 32.0450 - val_loss: 247.4264 - val_mae: 29.2047
Epoch 9/2

,Model,Feature Set,MAE (sec),MAE (min),RMSE (sec),RMSE (min),R²
0,Persistence,Previous-stop delay only,38.679,0.645,60.564,1.009,0.9389
1,Ridge,Temporal + Schedule,37.195,0.620,57.224,0.954,0.9454
2,Ridge,Temporal + Spatial,36.485,0.608,54.599,0.910,0.9503
3,LightGBM,Temporal + Schedule,25.669,0.428,99.516,1.659,0.8349
4,LightGBM,Temporal + Spatial,23.560,0.393,98.075,1.635,0.8397
5,LSTM,Temporal + Schedule,29.246,0.487,83.818,1.397,0.8829
6,LSTM,Temporal + Spatial,21.568,0.359,45.091,0.752,0.9661



💡 你跑出的最新 LSTM 空间特征增益分析 (Ablation Gain):
• Temporal MAE: 29.246 s (0.487 min)  |  Spatial MAE: 21.568 s (0.359 min)
• Temporal RMSE: 83.818 s | Spatial RMSE: 45.091 s | R²: 0.9661
✅ Spatial 特征起到了正面作用！MAE 降低了 7.678 秒 (优化率: 26.25%)

✅ 对比总表已成功导出至: /content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed/master_model_ablation_comparison.csv


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import r2_score
from sklearn.preprocessing import RobustScaler
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import LSTM, BatchNormalization, Dense, Dropout
from tensorflow.keras.models import Sequential

# ---------------------------------------------------------
# 0. 目录配置
# ---------------------------------------------------------
MODELS_DIR = './models' if 'MODELS_DIR' not in globals() else MODELS_DIR
DATA_DIR = './data' if 'DATA_DIR' not in globals() else DATA_DIR
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ---------------------------------------------------------
# 1. 特征定义 (严格与队友 Temporal 与 Spatial 特征清单对齐)
# ---------------------------------------------------------
FEATURE_COLS_TEMPORAL = [
    'delay_lag_1',
    'delay_lag_2',
    'delay_diff_prev',
    'scheduled_arrival_time_sfm',
    'hour_sin',
    'hour_cos',
    'day_of_week',
    'is_weekend',
    'is_morning_peak',
    'is_evening_peak',
    'stop_sequence',
    'dist_traveled',
]

FEATURE_COLS_SPATIAL = FEATURE_COLS_TEMPORAL + [
    'network_length_m',
    'num_traffic_signals',
    'num_intersections',
    'bus_lane_ratio',
    'has_bus_priority',
]

TARGET_COL = 'observed_arrival_delay'

# 检查特征列完整性
missing_cols = [col for col in FEATURE_COLS_SPATIAL if col not in df.columns]
if missing_cols:
    print(f'❌ 缺失列: {missing_cols}')
else:
    print(
        f'✅ 特征列校验 100% 通过！纯时序特征数 = {len(FEATURE_COLS_TEMPORAL)} 维 |'
        f' 时空增强特征数 = {len(FEATURE_COLS_SPATIAL)} 维'
    )

# ---------------------------------------------------------
# 2. 严格按队友要求的 Temporal Split (Jan 1–18 / Jan 19–23 / Jan 24–31)
# ---------------------------------------------------------
train_df = df[(df['date'] >= 20240101) & (df['date'] <= 20240118)].copy()
val_df   = df[(df['date'] >= 20240119) & (df['date'] <= 20240123)].copy()
test_df  = df[(df['date'] >= 20240124) & (df['date'] <= 20240131)].copy()

print(f"📊 数据划分完成: Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")


# ---------------------------------------------------------
# 3. 严谨 3D 序列构建函数 (仅在 Train fit，防泄露 + 按 ['date', 'trip_id', 'direction_id'] 分组)
# ---------------------------------------------------------
def prepare_3d_tensors(feature_cols, sequence_length=3):
    scaler = RobustScaler()
    train_scaled = train_df.copy()
    val_scaled   = val_df.copy()
    test_scaled  = test_df.copy()

    # 严谨拟合：仅在 Train 上 fit，在 Val 和 Test 上 transform
    train_scaled[feature_cols] = scaler.fit_transform(train_scaled[feature_cols])
    val_scaled[feature_cols]   = scaler.transform(val_scaled[feature_cols])
    test_scaled[feature_cols]  = scaler.transform(test_scaled[feature_cols])

    def build_correct_sequences(df_data):
        X, y = [], []
        # 队友核心要求：序列分组统一使用 ['date', 'trip_id', 'direction_id']
        group_cols = ['date', 'trip_id', 'direction_id']

        for _, group in df_data.groupby(group_cols):
            group = group.sort_values('stop_sequence')
            features = group[feature_cols].values
            targets = group[TARGET_COL].values

            n_samples = len(features)
            if n_samples > 0:
                for i in range(n_samples):
                    if i < sequence_length - 1:
                        pad_len = sequence_length - (i + 1)
                        pad_features = np.tile(features[0], (pad_len, 1))
                        seq_features = np.vstack([pad_features, features[: i + 1]])
                    else:
                        seq_features = features[i - sequence_length + 1 : i + 1]

                    X.append(seq_features)
                    y.append(targets[i])

        return np.array(X), np.array(y)

    X_tr, y_tr = build_correct_sequences(train_scaled)
    X_va, y_va = build_correct_sequences(val_scaled)
    X_te, y_te = build_correct_sequences(test_scaled)

    return X_tr, y_tr, X_va, y_va, X_te, y_te


# ---------------------------------------------------------
# 4. 标准化 LSTM 训练与评估（包含：剔除 > 10 min 极端延误评估）
# ---------------------------------------------------------
def train_and_evaluate(X_tr, y_tr, X_va, y_va, X_te, y_te, exp_name, save_filename, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.backend.clear_session()

    model = Sequential([
        LSTM(
            64,
            return_sequences=True,
            input_shape=(X_tr.shape[1], X_tr.shape[2]),
        ),
        BatchNormalization(),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        BatchNormalization(),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1, activation='linear'),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.Huber(delta=10.0),
        metrics=['mae'],
    )

    save_path = os.path.join(MODELS_DIR, save_filename)
    callbacks = [
        EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True
        ),
        ModelCheckpoint(save_path, monitor='val_loss', save_best_only=True),
    ]

    print('\n' + '=' * 55)
    print(f'🚀 开始训练实验: {exp_name}')
    print(f'特征输入维度: {X_tr.shape[2]} 维 | 训练集样本数: {X_tr.shape[0]} | 验证集样本数: {X_va.shape[0]}')
    print('=' * 55)

    model.fit(
        X_tr,
        y_tr,
        validation_data=(X_va, y_va),
        epochs=25,
        batch_size=128,
        callbacks=callbacks,
        verbose=1,
    )

    # 1. 预测全量测试集
    y_pred_all = model.predict(X_te).flatten()

    # 2. 剔除极端值过滤核心逻辑：只保留绝对延误 <= 600 秒（10 分钟）的 Normal Operations 样本
    # 注意：真实值和预测值都要按照真实绝对延误 <= 600 秒提取
    normal_mask = np.abs(y_te) <= 600  # 10 分钟 = 600 秒

    y_te_norm = y_te[normal_mask]
    y_pred_norm = y_pred_all[normal_mask]

    n_norm = np.sum(normal_mask)
    n_total = len(y_te)
    print(f"\n🔍 Normal-operation samples: {n_norm} / {n_total} ({n_norm/n_total*100:.1f}%)")

    # 计算剔除异常值后的指标
    mae_norm = np.mean(np.abs(y_te_norm - y_pred_norm))
    rmse_norm = np.sqrt(np.mean((y_te_norm - y_pred_norm) ** 2))
    r2_norm = r2_score(y_te_norm, y_pred_norm)

    return mae_norm, rmse_norm, r2_norm, n_norm, n_total


# ---------------------------------------------------------
# 5. 执行两次独立消融实验 (同一 Seed，保证模型独立重新初始化)
# ---------------------------------------------------------
# 实验 3a: LSTM Temporal Baseline (12 维)
X_tr_temp, y_tr_temp, X_va_temp, y_va_temp, X_te_temp, y_te_temp = prepare_3d_tensors(
    FEATURE_COLS_TEMPORAL
)
mae_norm_temp, rmse_norm_temp, r2_norm_temp, n_norm, n_total = train_and_evaluate(
    X_tr_temp, y_tr_temp,
    X_va_temp, y_va_temp,
    X_te_temp, y_te_temp,
    'Model 3a: Deep LSTM Temporal Baseline (12 features)',
    'best_lstm_temporal_12d.keras',
    seed=42
)

# 实验 3b: LSTM Spatial-Enhanced (17 维)
X_tr_spat, y_tr_spat, X_va_spat, y_va_spat, X_te_spat, y_te_spat = prepare_3d_tensors(
    FEATURE_COLS_SPATIAL
)
mae_norm_spat, rmse_norm_spat, r2_norm_spat, _, _ = train_and_evaluate(
    X_tr_spat, y_tr_spat,
    X_va_spat, y_va_spat,
    X_te_spat, y_te_spat,
    'Model 3b: Deep LSTM Spatial-Enhanced (17 features)',
    'best_lstm_spatial_17d.keras',
    seed=42
)

# ---------------------------------------------------------
# 6. 100% 对齐队友截图 2 (Normal-operation samples) 格式的汇总表
# ---------------------------------------------------------
master_data_normal = [
    {
        'Model': 'Persistence',
        'MAE (sec)': 38.185,
        'MAE (min)': 0.636,
        'RMSE (sec)': 56.679,
        'RMSE (min)': 0.945,
        'R²': 0.8333,
    },
    {
        'Model': 'Ridge Temporal',
        'MAE (sec)': 36.606,
        'MAE (min)': 0.610,
        'RMSE (sec)': 52.981,
        'RMSE (min)': 0.883,
        'R²': 0.8543,
    },
    {
        'Model': 'Ridge Spatial',
        'MAE (sec)': 35.939,
        'MAE (min)': 0.599,
        'RMSE (sec)': 50.314,
        'RMSE (min)': 0.839,
        'R²': 0.8686,
    },
    {
        'Model': 'LightGBM Temporal',
        'MAE (sec)': 22.751,
        'MAE (min)': 0.379,
        'RMSE (sec)': 32.749,
        'RMSE (min)': 0.546,
        'R²': 0.9443,
    },
    {
        'Model': 'LightGBM Spatial',
        'MAE (sec)': 20.664,
        'MAE (min)': 0.344,
        'RMSE (sec)': 30.620,
        'RMSE (min)': 0.510,
        'R²': 0.9513,
    },
    {
        'Model': 'LSTM Temporal',
        'MAE (sec)': round(mae_norm_temp, 3),
        'MAE (min)': round(mae_norm_temp / 60, 3),
        'RMSE (sec)': round(rmse_norm_temp, 3),
        'RMSE (min)': round(rmse_norm_temp / 60, 3),
        'R²': round(r2_norm_temp, 4),
    },
    {
        'Model': 'LSTM Spatial',
        'MAE (sec)': round(mae_norm_spat, 3),
        'MAE (min)': round(mae_norm_spat / 60, 3),
        'RMSE (sec)': round(rmse_norm_spat, 3),
        'RMSE (min)': round(rmse_norm_spat / 60, 3),
        'R²': round(r2_norm_spat, 4),
    },
]

df_master_normal = pd.DataFrame(master_data_normal)

print('\n' + '=' * 65)
print(f'Normal-operation samples: {n_norm} / {n_total} ({n_norm/n_total*100:.1f}%)')
print('=' * 65)
display(df_master_normal)

# 计算 LSTM 空间特征在正常样本下的增益
mae_diff_norm = mae_norm_temp - mae_norm_spat
mae_gain_norm = (mae_diff_norm / mae_norm_temp) * 100

print('\n💡 剔除极端延误后（Normal Operation），你的 LSTM 空间特征增益:')
print(f'• Temporal MAE: {mae_norm_temp:.3f} s  |  Spatial MAE: {mae_norm_spat:.3f} s')
print(f'• Temporal RMSE: {rmse_norm_temp:.3f} s |  Spatial RMSE: {rmse_norm_spat:.3f} s | R²: {r2_norm_spat:.4f}')

if mae_diff_norm > 0:
    print(f'✅ 剔除极端干扰后，Spatial 特征效果提升：MAE 降低了 {mae_diff_norm:.3f} 秒 (优化率: {mae_gain_norm:.2f}%)')

# 保存导出的预测汇总 CSV
df_master_normal.to_csv(os.path.join(DATA_DIR, 'normal_operation_ablation_comparison.csv'), index=False)
print(f"\n✅ 诊断表已导出保存至: {os.path.join(DATA_DIR, 'normal_operation_ablation_comparison.csv')}")

✅ 特征列校验 100% 通过！纯时序特征数 = 12 维 | 时空增强特征数 = 17 维
📊 数据划分完成: Train=115164 | Val=27666 | Test=52694


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



🚀 开始训练实验: Model 3a: Deep LSTM Temporal Baseline (12 features)
特征输入维度: 12 维 | 训练集样本数: 115164 | 验证集样本数: 27666
Epoch 1/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - loss: 738.9549 - mae: 78.6659 - val_loss: 436.9688 - val_mae: 48.3794
Epoch 2/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - loss: 415.2664 - mae: 46.2235 - val_loss: 374.1446 - val_mae: 42.0682
Epoch 3/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 388.1055 - mae: 43.4964 - val_loss: 356.5708 - val_mae: 40.2748
Epoch 4/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 369.0460 - mae: 41.5756 - val_loss: 331.2923 - val_mae: 37.7292
Epoch 5/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 354.1723 - mae: 40.0797 - val_loss: 324.9887 - val_mae: 37.0971
Epoch 6/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - loss: 345.0958 - mae: 39.1625 - val_loss: 306.1062 - val_mae: 35.2034
Epoch 7/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - loss: 334.9940 - mae: 38.1452 - val_loss: 295.3915 - val_mae: 34.0953
Epoch 8/25
900/9

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


900/900 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - loss: 696.1438 - mae: 74.3551 - val_loss: 349.7433 - val_mae: 39.5177
Epoch 2/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - loss: 348.8302 - mae: 39.5258 - val_loss: 288.5429 - val_mae: 33.3877
Epoch 3/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 325.9869 - mae: 37.2224 - val_loss: 275.8586 - val_mae: 32.1239
Epoch 4/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 308.0157 - mae: 35.4053 - val_loss: 262.7188 - val_mae: 30.7687
Epoch 5/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 295.5799 - mae: 34.1423 - val_loss: 252.7246 - val_mae: 29.7172
Epoch 6/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 288.5242 - mae: 33.4315 - val_loss: 249.2398 - val_mae: 29.3804
Epoch 7/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 281.7923 - mae: 32.7494 - val_loss: 264.0964 - val_mae: 30.8673
Epoch 8/25
900/900 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 274.8384 - mae: 32.0450 - val_loss: 247.4264 - val_mae: 29.2047
Epoch 9/25


,Model,MAE (sec),MAE (min),RMSE (sec),RMSE (min),R²
0,Persistence,38.185,0.636,56.679,0.945,0.8333
1,Ridge Temporal,36.606,0.610,52.981,0.883,0.8543
2,Ridge Spatial,35.939,0.599,50.314,0.839,0.8686
3,LightGBM Temporal,22.751,0.379,32.749,0.546,0.9443
4,LightGBM Spatial,20.664,0.344,30.620,0.510,0.9513
5,LSTM Temporal,26.835,0.447,38.705,0.645,0.9223
6,LSTM Spatial,19.921,0.332,30.181,0.503,0.9527



💡 剔除极端延误后（Normal Operation），你的 LSTM 空间特征增益:
• Temporal MAE: 26.835 s  |  Spatial MAE: 19.921 s
• Temporal RMSE: 38.705 s |  Spatial RMSE: 30.181 s | R²: 0.9527
✅ 剔除极端干扰后，Spatial 特征效果提升：MAE 降低了 6.914 秒 (优化率: 25.76%)

✅ 诊断表已导出保存至: /content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed/normal_operation_ablation_comparison.csv
